# What's new in MissingPatterns.jl 0.7

[MissingPatterns.jl](https://github.com/dantebertuzzi/MissingPatterns.jl) 0.7 adds a formal test
of the question every other view only hints at — *is this missing at random?* — a way to see
*what* the missingness depends on, a finder for absences coded as values, and three things that
make the existing views more honest: diffs that pair records by key, pattern tables ordered
by structure, and grouped plots that stop drawing the grouping column.

The table is the one from
[`getting-started.ipynb`](https://colab.research.google.com/github/dantebertuzzi/MissingPatterns.jl/blob/main/notebooks/getting-started.ipynb),
so the holes have known causes and every result below can be checked against them.

Full changelog: <https://github.com/dantebertuzzi/MissingPatterns.jl/blob/main/CHANGELOG.md>

**On Colab, pick the Julia runtime first:** *Runtime ▸ Change runtime type ▸ Julia*.

In [ ]:
using Pkg
Pkg.add(["MissingPatterns", "DataFrames"])
Pkg.status("MissingPatterns")   # needs 0.7 or later

In [ ]:
using MissingPatterns, DataFrames, Dates, Random

## The table

A thousand records, built with three mechanisms that do not show in the table itself:

- **income** is refused more often in the south and by respondents over 70 — missing *because
  of* values in other columns;
- **lab** and **weight** come from the same clinical visit, which was skipped more often before
  2021 — missing because of the date, but not of any measured value;
- **schooling** codes "ignored" as `99` instead of `missing`.

This version adds an `id` column, used further down to pair two releases record by record.

In [1]:
Random.seed!(20260911)
n = 1000

region    = rand(["north", "south", "east", "west"], n)
date      = Date(2019, 1, 1) .+ Day.(rand(0:2_190, n))
age       = rand(18:89, n)
schooling = rand([0, 1, 2, 3, 99], n)

income = Vector{Union{Missing,Int}}(round.(Int, 1500 .+ 120 .* (age .- 18) .+ 800 .* randn(n)))
lab    = Vector{Union{Missing,Float64}}(round.(3.5 .+ 0.9 .* randn(n); digits = 2))
weight = Vector{Union{Missing,Float64}}(round.(60 .+ 12 .* randn(n); digits = 1))

for i in 1:n
    # refusal to state income: more in the south, more among the oldest
    rand() < 0.10 + 0.25 * (region[i] == "south") + 0.20 * (age[i] > 70) && (income[i] = missing)

    # a skipped visit takes the lab result, and usually the weight, with it
    if rand() < 0.30 + 0.35 * (date[i] < Date(2021, 1, 1))
        lab[i] = missing
        rand() < 0.8 && (weight[i] = missing)
    end
end

df = DataFrame(; id = 1:n, region, date, age, schooling, income, lab, weight)
first(df, 6)

Row,id,region,date,age,schooling,income,lab,weight
,Int64,String,Date,Int64,Int64,Int64?,Float64?,Float64?
1,1,west,2022-12-30,43,1,5596,1.5,95.2
2,2,east,2024-11-09,84,3,missing,missing,50.9
3,3,west,2022-12-19,42,0,4518,missing,67.9
4,4,east,2022-10-13,64,2,missing,missing,missing
5,5,south,2020-03-27,56,0,missing,missing,missing
6,6,east,2019-10-05,85,3,8935,missing,58.6


## Is it missing completely at random?

`missingmcar` runs Little's (1988) test. It estimates the means and covariance of the numeric
columns by EM, then asks whether the rows of each missingness pattern have the means those
estimates predict. Under MCAR they should: which values happen to be missing would tell you
nothing about them. A small p-value says the patterns differ systematically.

`schooling` is left out — `99` is a code, not a number of years, and the test assumes continuous
columns.

In [2]:
r = missingmcar(df; columns = [:age, :income, :lab, :weight])
(statistic = r.statistic, df = r.df, pvalue = r.pvalue, npatterns = r.npatterns)

(statistic = 22.451605203461007, df = 11, pvalue = 0.02109895459042477, npatterns = 6)

MCAR is rejected at 5%. The income refusals depend on age, so the rows missing income are
older than the rest, and the test sees that in the pattern means.

Now drop `income` and test the clinical visit on its own:

In [3]:
r = missingmcar(df; columns = [:age, :lab, :weight])
(statistic = r.statistic, df = r.df, pvalue = r.pvalue, npatterns = r.npatterns)

(statistic = 1.1492280222941513, df = 3, pvalue = 0.7652054652468223, npatterns = 3)

No evidence against MCAR — and yet `lab` and `weight` are *not* missing completely at random:
the visit was skipped more often before 2021. The test cannot see that, because nothing it was
given depends on the date: ages and lab values are the same in every year.

That is the general limit of the test. A small p-value is evidence against MCAR; a large one only
says the columns you tested show no trace of the mechanism. It never proves MCAR, and it cannot
tell MAR from MNAR. Read it next to the views below rather than instead of them — `missingpredictors`, two
sections down, finds the date.

The result also carries the EM estimates, which are the maximum-likelihood mean and covariance
under the normal model — available-case means are biased whenever MCAR fails:

In [4]:
r = missingmcar(df; columns = [:age, :income, :lab, :weight])
DataFrame(column        = r.columns,
          em_mean       = round.(r.mu; digits = 1),
          observed_mean = [round(sum(skipmissing(df[!, c])) / count(!ismissing, df[!, c]); digits = 1)
                           for c in r.columns])

Row,column,em_mean,observed_mean
,Symbol,Float64,Float64
1,age,54.0,54.0
2,income,5836.2,5653.4
3,lab,3.5,3.4
4,weight,60.2,60.2


The observed mean of `income` is lower than the EM estimate: the people who refused were the
older ones, and older respondents earn more here.

## Absence that does not say `missing`

The MCAR test above left `schooling` out by hand, because its `99` means "ignored". On a real
file nobody tells you that. `missingsentinels` proposes the values that probably encode absence,
each with the evidence for it — here, a code that sits far above every other value of its
column:

In [5]:
found = missingsentinels(df)
DataFrame(found)

Row,column,value,n,pct,reason
,Symbol,Any,Int64,Float64,String
1,schooling,99,206,20.6,above every other value (max 3)


These are candidates for review, not verdicts: a numeric code is only proposed when the data
back it, and text only when it is blank or reads as absent. Keep the ones that are codes, and
`sentinelisna` turns them into the per-column `isna` that every function takes:

In [6]:
isna = sentinelisna(found)
missingsummary(df; isna, color = :never)

 column     type       missing        %  distribution
 lab        Float64        442   44.20%  ▄▄▃▄▄▄▄▅▄▅▃▄▅▃▃▃▄▄▅▄
 weight     Float64        351   35.10%  ▄▃▃▃▃▃▃▄▃▄▃▄▄▃▃▃▄▃▄▂
 schooling  Int64          206   20.60%  ▂▁▂▃▃▂▂▂▂▂▃▂▂▃▃▂▂▁▁▃
 income     Int64          202   20.20%  ▂▂▂▂▁▂▃▂▃▂▂▃▁▂▂▂▁▂▂▂
 id         Int64            0    0.00%                      
 region     String           0    0.00%                      
 date       Date             0    0.00%                      
 age        Int64            0    0.00%                      
 1201 missing of 8000 cells (15.01%) across 8 columns ┊ bins of 50 rows


## What the missingness depends on

`missingmcar` says *whether* the missingness is related to the data. `missingpredictors` says *to
what*: for one column, every other column is compared between the rows where it is missing and
the rows where it is present — Welch's t-test for numbers and dates, a χ² test for the rest —
most significant first.

In [7]:
pred(col) = select(DataFrame(missingpredictors(df, col; isna)),
                  :column, :test, :pvalue, :effect, :detail)
pred(:income)

Row,column,test,pvalue,effect,detail
,Symbol,Symbol,Float64,Float64,String
1,region,chisq,2.17923e-8,0.196299,south: 42.1% of rows where missing vs 20.9% where present
2,age,welch_t,1.8201e-5,0.352463,mean 60 where missing vs 52.47 where present
3,date,welch_t,0.241597,0.0920695,mean 2022-02-18 where missing vs 2021-12-21 where present
4,id,welch_t,0.427811,-0.0624609,mean 486.1 where missing vs 504.1 where present
5,schooling,welch_t,0.699816,0.0332727,mean 1.509 where missing vs 1.472 where present
6,lab,welch_t,0.756499,-0.0303123,mean 3.429 where missing vs 3.456 where present
7,weight,welch_t,0.870057,0.0150936,mean 60.32 where missing vs 60.15 where present


Both mechanisms built into `income` come out on top: refusals are over-represented in the south
and among older respondents. Nothing else is related, which is also true by construction.

`effect` is the standardized mean difference for numbers (0.2 small, 0.5 medium, 0.8 large) and
Cramér's V for categories — read it next to the p-value, since with many columns a few unrelated
ones will cross 5% by chance.

Now the clinical visit, which the MCAR test could not see through:

In [8]:
pred(:lab)

Row,column,test,pvalue,effect,detail
,Symbol,Symbol,Float64,Float64,String
1,date,welch_t,1.69141e-19,-0.592001,mean 2021-06-12 where missing vs 2022-06-12 where present
2,schooling,welch_t,0.259033,0.0806608,mean 1.53 where missing vs 1.44 where present
3,age,welch_t,0.321607,-0.0630959,mean 53.26 where missing vs 54.58 where present
4,income,welch_t,0.575295,-0.0398509,mean 5597 where missing vs 5700 where present
5,region,chisq,0.61687,0.0423216,east: 27.1% of rows where missing vs 24.4% where present
6,id,welch_t,0.814952,-0.0149043,mean 498.1 where missing vs 502.4 where present
7,weight,welch_t,0.918578,-0.0106036,mean 60.08 where missing vs 60.2 where present


`date` comes first, by a wide margin: `lab` is missing on records a year older on average. The
MCAR test was given ages and lab values, none of which depend on the date; compared column by
column, the date itself gives the mechanism away. Whatever imputes `lab` has to condition on it.

## Pattern tables ordered by structure

`missingpatterns` now takes the same `order` keyword as `plotmissing`. With `:cluster`, columns
that go missing together sit side by side, so each pattern's holes read as one block instead of
scattered marks. The order comes from the pattern table already computed, at no extra pass over
the data.

In [9]:
missingpatterns(df; order = :cluster, color = :never)

┏━━━━━━━━━┳━━━━━━━━━┳━━━━━━━━━┳━━━━━━━━━┳━━━━━━━━━┳━━━━━━━━━┳━━━━━━━━━┳━━━━━━━━━┳━━━━━━━━━┳━━━━━━━━━┳━━━━━━━━━┓
┃   lab   ┃  weig…  ┃  inco…  ┃   id    ┃  regi…  ┃  date   ┃   age   ┃  scho…  ┃    n    ┃    %    ┃  freq   ┃
┣━━━━━━━━━╋━━━━━━━━━╋━━━━━━━━━╋━━━━━━━━━╋━━━━━━━━━╋━━━━━━━━━╋━━━━━━━━━╋━━━━━━━━━╋━━━━━━━━━╋━━━━━━━━━╋━━━━━━━━━┫
┃  ░░░░░  ┃  ░░░░░  ┃  ░░░░░  ┃  ░░░░░  ┃  ░░░░░  ┃  ░░░░░  ┃  ░░░░░  ┃  ░░░░░  ┃   436   ┃  43.6%  ┃ ███████ ┃
┃  █████  ┃  █████  ┃  ░░░░░  ┃  ░░░░░  ┃  ░░░░░  ┃  ░░░░░  ┃  ░░░░░  ┃  ░░░░░  ┃   284   ┃  28.4%  ┃ █████   ┃
┃  ░░░░░  ┃  ░░░░░  ┃  █████  ┃  ░░░░░  ┃  ░░░░░  ┃  ░░░░░  ┃  ░░░░░  ┃  ░░░░░  ┃   122   ┃  12.2%  ┃ ██      ┃
┃  █████  ┃  ░░░░░  ┃  ░░░░░  ┃  ░░░░░  ┃  ░░░░░  ┃  ░░░░░  ┃  ░░░░░  ┃  ░░░░░  ┃   78    ┃  7.8%   ┃ █       ┃
┃  █████  ┃  █████  ┃  █████  ┃  ░░░░░  ┃  ░░░░░  ┃  ░░░░░  ┃  ░░░░░  ┃  ░░░░░  ┃   67    ┃  6.7%   ┃ █       ┃
┃  █████  ┃  ░░░░░  ┃  █████  ┃  ░░░░░  ┃  ░░░░░  ┃  ░░░░░  ┃  ░░░░░  ┃  ░░░░░  ┃   13    ┃  1.3%   ┃   

`lab` and `weight` end up adjacent, and the patterns where both are missing read as a single
run.

## Grouping without restating the group

Grouped with `by`, the heatmap used to draw the grouping column too — present in every group,
missing only in `∅`, which says nothing new and dilutes the totals. It is now left out; pass
`exclude_by = false` to keep it.

In [10]:
plotmissing(df; by = :region, layout = :classic, color = :never)

┏━━━━━┳━━━━━━━━━┳━━━━━━━━━┳━━━━━━━━━┳━━━━━━━━━┳━━━━━━━━━┳━━━━━━━━━┳━━━━━━━━━┓
┃     ┃    0%   ┃    0%   ┃    0%   ┃    0%   ┃   20%   ┃   44%   ┃   35%   ┃
┣━━━━━╋━━━━━━━━━╋━━━━━━━━━╋━━━━━━━━━╋━━━━━━━━━╋━━━━━━━━━╋━━━━━━━━━╋━━━━━━━━━┫
┃ row ┃   id    ┃  date   ┃   age   ┃  scho…  ┃  inco…  ┃   lab   ┃  weig…  ┃
┣━━━━━╋━━━━━━━━━╋━━━━━━━━━╋━━━━━━━━━╋━━━━━━━━━╋━━━━━━━━━╋━━━━━━━━━╋━━━━━━━━━┫
┃east ┃  ░░░░░  ┃  ░░░░░  ┃  ░░░░░  ┃  ░░░░░  ┃  ░░░░░  ┃  ▓▓▓▓▓  ┃  ▓▓▓▓▓  ┃
┃north┃  ░░░░░  ┃  ░░░░░  ┃  ░░░░░  ┃  ░░░░░  ┃  ▒▒▒▒▒  ┃  ▓▓▓▓▓  ┃  ▓▓▓▓▓  ┃
┃south┃  ░░░░░  ┃  ░░░░░  ┃  ░░░░░  ┃  ░░░░░  ┃  ▓▓▓▓▓  ┃  ▓▓▓▓▓  ┃  ▓▓▓▓▓  ┃
┃west ┃  ░░░░░  ┃  ░░░░░  ┃  ░░░░░  ┃  ░░░░░  ┃  ▒▒▒▒▒  ┃  ▓▓▓▓▓  ┃  ▓▓▓▓▓  ┃
┗━━━━━┻━━━━━━━━━┻━━━━━━━━━┻━━━━━━━━━┻━━━━━━━━━┻━━━━━━━━━┻━━━━━━━━━┻━━━━━━━━━┛

MissingPatterns.Analysis: 1000 × 7 table
 Grouping:    by region → 4×7 cells
 Missing (count):             995               ┊ Missing (%):          14.21%
 Present (count):            6005               ┊ Present (%): 

The south row carries the extra income refusals. The summary counts the columns actually
drawn.

## Row labels in HTML

`missinghtml` — and `missingreport`, which renders through it in a notebook — now labels the
rows of the grid. Labels are on whenever `by` is set, and `show_row_range = true` turns them on
for positional blocks.

In [11]:
missingreport(df; by = :date, period = :year, order = :cluster)

┏━━━━━┳━━━━━━━━━┳━━━━━━━━━┳━━━━━━━━━┳━━━━━━━━━┳━━━━━━━━━┳━━━━━━━━━┳━━━━━━━━━┓
┃     ┃   44%   ┃   35%   ┃   20%   ┃    0%   ┃    0%   ┃    0%   ┃    0%   ┃
┣━━━━━╋━━━━━━━━━╋━━━━━━━━━╋━━━━━━━━━╋━━━━━━━━━╋━━━━━━━━━╋━━━━━━━━━╋━━━━━━━━━┫
┃ row ┃   lab   ┃  weig…  ┃  inco…  ┃   id    ┃  regi…  ┃   age   ┃  scho…  ┃
┣━━━━━╋━━━━━━━━━╋━━━━━━━━━╋━━━━━━━━━╋━━━━━━━━━╋━━━━━━━━━╋━━━━━━━━━╋━━━━━━━━━┫
┃2019 ┃  █████  ┃  █████  ┃  ▒▒▒▒▒  ┃  ░░░░░  ┃  ░░░░░  ┃  ░░░░░  ┃  ░░░░░  ┃
┃2020 ┃  █████  ┃  █████  ┃  ▒▒▒▒▒  ┃  ░░░░░  ┃  ░░░░░  ┃  ░░░░░  ┃  ░░░░░  ┃
┃2021 ┃  ▓▓▓▓▓  ┃  ▒▒▒▒▒  ┃  ▒▒▒▒▒  ┃  ░░░░░  ┃  ░░░░░  ┃  ░░░░░  ┃  ░░░░░  ┃
┃2022 ┃  ▓▓▓▓▓  ┃  ▒▒▒▒▒  ┃  ▒▒▒▒▒  ┃  ░░░░░  ┃  ░░░░░  ┃  ░░░░░  ┃  ░░░░░  ┃
┃2023 ┃  ▓▓▓▓▓  ┃  ▒▒▒▒▒  ┃  ▒▒▒▒▒  ┃  ░░░░░  ┃  ░░░░░  ┃  ░░░░░  ┃  ░░░░░  ┃
┃2024 ┃  ▓▓▓▓▓  ┃  ▒▒▒▒▒  ┃  ▒▒▒▒▒  ┃  ░░░░░  ┃  ░░░░░  ┃  ░░░░░  ┃  ░░░░░  ┃
┗━━━━━┻━━━━━━━━━┻━━━━━━━━━┻━━━━━━━━━┻━━━━━━━━━┻━━━━━━━━━┻━━━━━━━━━┻━━━━━━━━━┛

MissingPatterns.Analysis: 1000 × 7 table
 Grouping:    by date (year) → 6×7 cells
 Missing (count):             995               ┊ Missing (%):          14.21%
 Present (count):            6005               ┊ Present (%):          85.79%
 Progress Bar:     [██████████████████████████████████████████████████████████]

The skipped visits fade after 2020, which is the date mechanism that the MCAR test above could
not see.

## Comparing two releases by key

`plotmissingdiff` compares two versions of a table. It used to need them aligned row for row;
`on` pairs rows by a key column instead, so a later release that was re-sorted, filtered or
appended to still compares record by record.

The second release below is shuffled, loses 50 records, gains 30 new ones, and has had a round of
data cleaning: 120 lab results recovered, and 15 weights lost.

In [12]:
Random.seed!(7)
release2 = df[shuffle(51:n), :]                 # re-sorted, records 1–50 gone

recovered = findall(ismissing, release2.lab)[1:120]
release2.lab[recovered] .= round.(3.5 .+ 0.9 .* randn(120); digits = 2)
lost = findall(!ismissing, release2.weight)[1:15]
release2.weight[lost] .= missing

newcomers = DataFrame(id = n+1:n+30, region = "north", date = Date(2025, 1, 1),
                      age = 40, schooling = 1, income = 4000, lab = 3.5, weight = 70.0)
release2 = vcat(release2, newcomers)
size(df), size(release2)

((1000, 8), (980, 8))

In [13]:
plotmissingdiff(df, release2; on = :id, color = :never)

┏━━━━━━━━━┳━━━━━━━━━┳━━━━━━━━━┳━━━━━━━━━┳━━━━━━━━━┳━━━━━━━━━┳━━━━━━━━━┓
┃ re… +0% ┃ da… +0% ┃ age +0% ┃ sc… +0% ┃ in… +0% ┃ l… -13% ┃ we… +2% ┃
┣━━━━━━━━━╋━━━━━━━━━╋━━━━━━━━━╋━━━━━━━━━╋━━━━━━━━━╋━━━━━━━━━╋━━━━━━━━━┫
┃  ·····  ┃  ·····  ┃  ·····  ┃  ·····  ┃  ·····  ┃  -----  ┃  +++++  ┃
┃  ·····  ┃  ·····  ┃  ·····  ┃  ·····  ┃  ·····  ┃  -----  ┃  ·····  ┃
┃  ·····  ┃  ·····  ┃  ·····  ┃  ·····  ┃  ·····  ┃  -----  ┃  ·····  ┃
┃  ·····  ┃  ·····  ┃  ·····  ┃  ·····  ┃  ·····  ┃  -----  ┃  +++++  ┃
┃  ·····  ┃  ·····  ┃  ·····  ┃  ·····  ┃  ·····  ┃  -----  ┃  +++++  ┃
┃  ·····  ┃  ·····  ┃  ·····  ┃  ·····  ┃  ·····  ┃  -----  ┃  ·····  ┃
┃  ·····  ┃  ·····  ┃  ·····  ┃  ·····  ┃  ·····  ┃  -----  ┃  ·····  ┃
┃  ·····  ┃  ·····  ┃  ·····  ┃  ·····  ┃  ·····  ┃  -----  ┃  +++++  ┃
┃  ·····  ┃  ·····  ┃  ·····  ┃  ·····  ┃  ·····  ┃  -----  ┃  +++++  ┃
┃  ·····  ┃  ·····  ┃  ·····  ┃  ·····  ┃  ·····  ┃  -----  ┃  +++++  ┃
┃  ·····  ┃  ·····  ┃  ·····  ┃  ·····  ┃  ·····  ┃  -----  ┃  ·

The summary counts exactly the edits made: 120 resolved, 15 introduced, over the 950 records
both releases share, with the 50 dropped and 30 added records reported apart. Compared by
position, the same two tables would line up different people and report noise.

## Faster, too

0.7 also makes the scans behind every view faster — ~20× on the heatmap's per-column tally, and
up to ~30× on pattern analysis of tables wider than 64 columns. Nothing to change in your code.


## References

The methods behind the views above; the [documentation](https://dantebertuzzi.github.io/MissingPatterns.jl/stable/references/) has the full list, and the [Methods](https://dantebertuzzi.github.io/MissingPatterns.jl/stable/methods/) page every formula.

- Little, R. J. A. (1988). A test of missing completely at random for multivariate data with missing values. *Journal of the American Statistical Association*, 83(404), 1198–1202. <https://doi.org/10.1080/01621459.1988.10478722>
- Dempster, A. P., Laird, N. M., & Rubin, D. B. (1977). Maximum likelihood from incomplete data via the EM algorithm. *Journal of the Royal Statistical Society: Series B*, 39(1), 1–22. <https://doi.org/10.1111/j.2517-6161.1977.tb01600.x>
- Rubin, D. B. (1976). Inference and missing data. *Biometrika*, 63(3), 581–592. <https://doi.org/10.1093/biomet/63.3.581>
- Little, R. J. A., & Rubin, D. B. (2019). *Statistical Analysis with Missing Data* (3rd ed.). Wiley. <https://doi.org/10.1002/9781119482260>
- van Buuren, S., & Groothuis-Oudshoorn, K. (2011). mice: Multivariate imputation by chained equations in R. *Journal of Statistical Software*, 45(3). <https://doi.org/10.18637/jss.v045.i03>
- Hahsler, M., Hornik, K., & Buchta, C. (2008). Getting things in order: An introduction to the R package seriation. *Journal of Statistical Software*, 25(3). <https://doi.org/10.18637/jss.v025.i03>
- Welch, B. L. (1947). The generalization of 'Student's' problem when several different population variances are involved. *Biometrika*, 34(1–2), 28–35. <https://doi.org/10.1093/biomet/34.1-2.28>
- Pearson, K. (1900). On the criterion that a given system of deviations from the probable in the case of a correlated system of variables is such that it can be reasonably supposed to have arisen from random sampling. *Philosophical Magazine*, 50(302), 157–175. <https://doi.org/10.1080/14786440009463897>
- Cramér, H. (1946). *Mathematical Methods of Statistics*. Princeton University Press. <https://doi.org/10.1515/9781400883868>
- Cohen, J. (1988). *Statistical Power Analysis for the Behavioral Sciences* (2nd ed.). Lawrence Erlbaum; reprinted 2013, Routledge. <https://doi.org/10.4324/9780203771587>

## Where to go next

- [Documentation](https://dantebertuzzi.github.io/MissingPatterns.jl/stable) — every keyword of
  every entry point
- [`getting-started.ipynb`](https://colab.research.google.com/github/dantebertuzzi/MissingPatterns.jl/blob/main/notebooks/getting-started.ipynb)
  — the full tour of the package
- [`obis-missingness.ipynb`](https://colab.research.google.com/github/dantebertuzzi/MissingPatterns.jl/blob/main/notebooks/obis-missingness.ipynb)
  — the same diagnostics on live marine biodiversity records